# 🚀 Document Refinery on Google Colab (GPU)

**GPU-accelerated document processing with intelligent triage and multi-strategy extraction**

### Features:
- 🎯 **Intelligent Triage**: Automatic document classification
- ⚡ **GPU Processing**: CUDA-accelerated OCR and image processing
- 🔄 **Multi-Strategy**: Fast Text → Layout → Vision escalation
- 📊 **Confidence-Gated**: Automatic strategy escalation when quality is low
- 🏗️ **Production Ready**: Complete logging, monitoring, and error handling

## 🔧 Setup Environment

### 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted successfully!")

### 2. Clone Document Refinery

In [ ]:
# Clone the repository
!git clone https://github.com/your-username/document-refinery.git /content/refinery
%cd /content/refinery

# Check GPU availability
import torch
print(f"🔥 GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### 3. Install Dependencies (GPU-Optimized)

In [ ]:
# Install PyTorch with CUDA support
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install OCR and document processing libraries
!pip install pdf2image pytesseract pdfplumber pymupdf docling pydantic pyyaml numpy

# Install system dependencies
!apt-get update && apt-get install -y tesseract-ocr tesseract-ocr-eng libtesseract-dev

# Install additional GPU libraries
!pip install opencv-python-headless

print("✅ All dependencies installed successfully!")

### 4. Upload Your Documents

In [ ]:
# Create data directory
!mkdir -p /content/refinery/data/raw

# Option 1: Upload files directly
from google.colab import files
print("📁 Upload your PDF files:")
uploaded = files.upload()

# Move uploaded files to data directory
for filename, file_data in uploaded.items():
    with open(f'/content/refinery/data/raw/{filename}', 'wb') as f:
        f.write(file_data)
    print(f"✅ Uploaded: {filename}")

## 🎯 Test GPU-Accelerated Processing

In [ ]:
# Test GPU setup
import sys
sys.path.append('/content/refinery/src')

# Import GPU-enabled extractor
from strategies.gpu_vision_extractor import GPUVisionExtractor
from models.document_models import DocumentProfile

# Test GPU extractor
gpu_extractor = GPUVisionExtractor(dpi=300, language='eng')

# Create test profile
test_profile = {
    'document_id': 'gpu_test',
    'file_path': '/content/refinery/data/raw/test.pdf',
    'total_pages': 5,
    'recommended_strategy': 'gpu_vision'
}

print("🧪 Testing GPU extractor setup...")
print(f"🔥 Device: {gpu_extractor.device}")
print(f"📊 GPU Available: {gpu_extractor.device.type == 'cuda'}")
print("✅ GPU extractor initialized successfully!")

## 🚀 Run Document Refinery with GPU

In [ ]:
# Run refinery with GPU acceleration
import os
import subprocess
import json
from pathlib import Path

# Set environment variables for GPU
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# Process all PDFs in data directory
data_dir = Path('/content/refinery/data/raw')
pdf_files = list(data_dir.glob('*.pdf'))

if pdf_files:
    print(f"📄 Found {len(pdf_files)} PDF files to process")
    
    # Process first file as test
    test_file = pdf_files[0]
    print(f"🔄 Processing: {test_file.name}")
    
    # Run refinery with GPU
    result = subprocess.run([
        'python3', 'refinery.py', 
        '--single', str(test_file),
        '--confidence-threshold', '0.7'
    ], capture_output=True, text=True, cwd='/content/refinery')
    
    print("📊 Processing Results:")
    print(result.stdout)
    
    # Show GPU utilization
    if torch.cuda.is_available():
        print(f"🔥 GPU Memory Used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
        print(f"💾 GPU Memory Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
else:
    print("⚠️ GPU not available, using CPU fallback")
else:
    print("📁 No PDF files found. Please upload documents first.")

## 📊 View Results

In [ ]:
# Display extraction results
import json
from pathlib import Path

# Show profiles
profiles_dir = Path('/content/refinery/.refinery/profiles')
if profiles_dir.exists():
    profile_files = list(profiles_dir.glob('*.json'))
    print(f"📋 Found {len(profile_files)} document profiles:")
    
    for profile_file in profile_files[-3:]:  # Show last 3
        with open(profile_file) as f:
            profile = json.load(f)
            print(f"📄 {profile_file.name}:")
            print(f"   Strategy: {profile.get('recommended_strategy', 'unknown')}")
            print(f"   Confidence: {profile.get('category_confidence', 0):.2f}")
            print(f"   Pages: {profile.get('total_pages', 0)}")
            print()

# Show extractions
extractions_dir = Path('/content/refinery/.refinery/extractions')
if extractions_dir.exists():
    extraction_files = list(extractions_dir.glob('*_extraction.json'))
    print(f"🔍 Found {len(extraction_files)} extraction results:")
    
    for extraction_file in extraction_files[-3:]:  # Show last 3
        with open(extraction_file) as f:
            extraction = json.load(f)
            print(f"📄 {extraction_file.name}:")
            print(f"   Strategy: {extraction.get('strategy_used', 'unknown')}")
            print(f"   Pages: {len(extraction.get('pages', []))}")
            print(f"   Tables: {extraction.get('extraction_metadata', {}).get('total_tables', 0)}")
            print(f"   Confidence: {extraction.get('extraction_metadata', {}).get('average_confidence', 0):.2f}")
            print()

## 💾 Save Results to Google Drive

All results are automatically saved to:
- `/content/refinery/.refinery/profiles/` - Document triage results
- `/content/refinery/.refinery/extractions/` - Extraction results
- `/content/refinery/.refinery/logs/` - Processing logs and ledger

To save to Google Drive:
```bash
# Copy results to Google Drive
!cp -r /content/refinery/.refinery /content/drive/MyDrive/refinery_results/
```

## 🎯 Performance Tips

### GPU Optimization:
- **Batch Processing**: Process multiple documents in parallel
- **Memory Management**: Monitor GPU memory usage
- **DPI Settings**: Lower DPI (200) for faster processing
- **Batch Size**: Process 5-10 pages at once

### Troubleshooting:
- **GPU Memory**: Reduce batch size if OOM errors
- **OCR Quality**: Adjust DPI and language settings
- **Processing Speed**: Use smaller documents for testing

### Expected Performance:
- **GPU Speed**: 5-10x faster than CPU
- **Memory Usage**: ~2-4GB for large documents
- **Throughput**: ~20-50 pages/second (GPU vs ~5 pages/second CPU)